# Guía 6 - Análisis geoespacial (capa ArcGIS)

In [2]:
#IMPORTACION DE LIBRERIAS

#Aqui se importan las librerias necesarias para manipular datos y las rutas

import pandas as pd
import sqlite3
import json
from pathlib import Path
import numpy as np
%pip install faker

print("Librerias importadas!!!!!!!!!!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 5.9 MB/s eta 0:00:00
Librerias importadas!!!!!!!!!!


In [3]:
# DEFINICION DE RUTAS DE TRABAJO

# Aqui se configuran las rutas para organizar los archivos del proyecto

BASE_DIR = Path("/content/")
DATA_RAW = BASE_DIR / "datos" / "crudos"
DATA_PROCESSED = BASE_DIR / "datos" / "procesados"
DATA_EXTERNAL = BASE_DIR / "datos" / "externos"
DOCUMENTOS = BASE_DIR / "documentos"

DOCUMENTOS.mkdir(parents=True, exist_ok=True)

print(f"Datos crudos: {DATA_RAW}")
print(f"Datos procesados: {DATA_PROCESSED}")
print(f"Datos externos: {DATA_EXTERNAL}")
print(f"Documentos: {DOCUMENTOS}")
print("Rutas de trabajo definidas correctamente.")

Datos crudos: /content/datos/crudos
Datos procesados: /content/datos/procesados
Datos externos: /content/datos/externos
Documentos: /content/documentos
Rutas de trabajo definidas correctamente.


In [77]:
# GUÍA 6 / FASE GEOESPACIAL: MAPA DE LAS 4 SEDES NACIONALES ARCGIS

import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import files

print("Generando capa para ArcGIS (4 Sedes ITCA-FEPADE)...")

GEO_DIR = BASE_DIR / "datos" / "geo"
GEO_DIR.mkdir(parents=True, exist_ok=True)

# 1. LECTURA DEL DATASET LIMPIO
df_limpio_fuente = pd.read_csv(DATA_PROCESSED / "dataset_limpio_Smart_Campus.csv")

# Asegurar tipos y recalculación directa
df_limpio_fuente["id_recurso"] = pd.to_numeric(df_limpio_fuente["id_recurso"], errors="coerce").astype(int)
df_limpio_fuente["estudiantes_presentes"] = pd.to_numeric(df_limpio_fuente["estudiantes_presentes"], errors="coerce").abs()
df_limpio_fuente["horas_uso"] = pd.to_numeric(df_limpio_fuente["horas_uso"], errors="coerce").fillna(2.0)

# Cruzar capacidad máxima desde el catálogo
df_limpio_fuente = df_limpio_fuente.drop(columns=["capacidad_maxima"], errors="ignore")
df_limpio_fuente = pd.merge(df_limpio_fuente, df_recursos_maestro[["id_recurso", "capacidad_maxima"]], on="id_recurso", how="left")

# Métricas recalculadas
df_limpio_fuente["horas_estudiante_calc"] = df_limpio_fuente["horas_uso"] * df_limpio_fuente["estudiantes_presentes"]
df_limpio_fuente["porcentaje_ocupacion_calc"] = (df_limpio_fuente["estudiantes_presentes"] / df_limpio_fuente["capacidad_maxima"]) * 100

# 2. AGREGACIÓN EQUITATIVA POR EDIFICIO
demanda_recursos = df_limpio_fuente.groupby("id_recurso").agg(
    total_sesiones=("id_registro", "count"),
    horas_estudiante_totales=("horas_estudiante_calc", "sum"),
    ocupacion_promedio=("porcentaje_ocupacion_calc", "mean"),
    total_estudiantes=("estudiantes_presentes", "sum")
).reset_index()

demanda_recursos["horas_estudiante_totales"] = demanda_recursos["horas_estudiante_totales"].round(2)
demanda_recursos["ocupacion_promedio"] = demanda_recursos["ocupacion_promedio"].round(2)

# 3. CRUCE ESPACIAL CON LAS COORDENADAS
capa_arcgis = pd.merge(df_recursos_maestro, demanda_recursos, on="id_recurso", how="left")

# 4. VALIDACIÓN TERRITORIAL PARA EL SALVADOR
mask_valida = (
    capa_arcgis['latitud'].between(13, 15) &
    capa_arcgis['longitud'].between(-91, -87)
)
print(f"Total edificios evaluados: {len(capa_arcgis)}")
print(f"Coordenadas válidas en El Salvador: {mask_valida.sum()}/20")

# 5. SEMÁFORO DE DEMANDA
capa_arcgis["alerta_demanda"] = np.where(
    capa_arcgis["ocupacion_promedio"] >= 80, "Alta Demanda (Rojo)",
    np.where(capa_arcgis["ocupacion_promedio"] >= 60, "Uso Moderado (Amarillo)", "Disponible (Verde)")
)

# 6. EXPORTACIÓN Y DESCARGA
ruta_csv_arcgis = GEO_DIR / "smartcampus_nacional_arcgis.csv"
capa_arcgis.to_csv(ruta_csv_arcgis, index=False, encoding='utf-8')

print(f"\nDescargando archivo nacional: {ruta_csv_arcgis.name}")
files.download(str(ruta_csv_arcgis))

Generando capa para ArcGIS (4 Sedes ITCA-FEPADE)...
Total edificios evaluados: 20
Coordenadas válidas en El Salvador: 20/20

Descargando archivo nacional: smartcampus_nacional_arcgis.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>